


# Вариант 4

In [7]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

f = pd.read_csv('fifa_players_stats.csv')
df = f[['Value(in Euro)','Overall', 'Potential', 'Age']].dropna()
print(df.describe(), "\n")

# Независимые переменные - рейтинг, потенциальный рейтинг и возраст
# Зависимая переменная - цена футболиста на рынке
X = f[['Overall', 'Potential', 'Age']]
y = f['Value(in Euro)']
alpha = 0.05

# Добавление константы для реализации линейной регрессии
X = sm.add_constant(X)

# Вычисление коэффициентов функции линейной регрессии y(x) / прогнозируемых весов
k = np.linalg.inv(X.T.dot(X)).dot(X.T).dot(y)
k = np.array([round(num, 2) for num in k])

# Предсказанные значения
y_predicted = X.dot(k)
deviation = y - y_predicted
residual_variance = np.sum(deviation**2) / (len(X) - 2)

# Оценка стандартных ошибок
cov_matrix = residual_variance * np.linalg.inv(X.T.dot(X))
std_errs = np.sqrt(np.diag(cov_matrix))
t_values = stats.t.ppf(1-alpha/2, len(df) - len(k))
confidence_intervals = np.column_stack((k - t_values * std_errs, k + t_values * std_errs))

# Оценка доли объясненной дисперсии и поиск коэффициента детерминации
real = np.sum((y - np.mean(y))**2)
predicted = np.sum((y_predicted - np.mean(y))**2)
R2 = predicted / real

# Вычисление стандартных ошибок коэффициентов
se = np.sqrt( np.diagonal( residual_variance * np.linalg.inv(X.T.dot(X)) ) )

# Вычисление доверительных интервалов для коэффициентов функции регрессии
t = np.abs(stats.t.ppf(alpha / 2, df=X.shape[0] - X.shape[1]))
intervals = []
for i in range(k.shape[0]):
    lower_bound = k[i] - t * se[i]
    upper_bound = k[i] + t * se[i]
    intervals.append((round(lower_bound, 2), round(upper_bound, 2)))

# Вычисление значения p-value
_, p_value_age = stats.ttest_ind(df['Value(in Euro)'], df['Age'])
_, p_value_overall = stats.ttest_ind(df['Value(in Euro)'], df['Overall'])
_, p_value_potential = stats.f_oneway(df['Value(in Euro)'], df['Potential'])

# Проверка гипотез

# Чем меньше возраст, тем больше цена
print("Цена - возраст:", round(p_value_age, 2))
if p_value_age < alpha:
    print("Гипотеза Н0 отвергается (цена не зависит от возраста) \n")
else:
    print("Гипотеза Н0 не отвергается (цена зависит от возраста) \n")

# Цена зависит от рейтинга
print("Цена - рейтинг:", round(p_value_overall, 2))
if p_value_overall < alpha:
    print("Гипотеза Н0 отвергается (цена не зависит от рейтинга) \n")
else:
    print("Гипотеза Н0 не отвергается (цена зависит от рейтинга) \n")

# Цена одновременно зависит от потенциального рейтинга и возраста
print("Цена - возраст, потенциальный рейтинг:", round(p_value_age, 2), ",", round(p_value_potential, 2))
if p_value_potential > alpha or p_value_age > alpha:
    print("Гипотеза Н0 отвергается (цена не одновременно зависит от потенциального рейтинга и возраста) \n")
else:
    print("Гипотеза Н0 не отвергается (цена одновременно зависит от потенциального рейтинга и возраста) \n")

# Вывод коэффициентов
print("Коэффициенты:")
print("Свободный член:", k[0])
print("Коэффициент для рейтинга, потенциального рейтинга и возраста:", k[1],",", k[2], "," , k[3], "\n")

# Вывод доверительных интервалов
print("Доверительные интервалы:")
print("Для свободного члена:", intervals[0])
print("Для рейтинга, потенциального рейтинга и возраста:", intervals[1],",", intervals[2], "," , intervals[3], "\n")

# Вывод параметров модели
print("Параметры модели:")
print("Остаточная дисперсия:", round(residual_variance, 2))
print("Коэффициент детерминации R^2:", round(R2, 3))

# Вывод результатов оценки модели
model = sm.OLS(y, X)
results = model.fit()
print(results.summary())


       Value(in Euro)       Overall     Potential           Age
count    1.853900e+04  18539.000000  18539.000000  18539.000000
mean     2.875461e+06     65.852042     71.016668     25.240412
std      7.635129e+06      6.788353      6.192866      4.718163
min      0.000000e+00     47.000000     48.000000     16.000000
25%      4.750000e+05     62.000000     67.000000     21.000000
50%      1.000000e+06     66.000000     71.000000     25.000000
75%      2.000000e+06     70.000000     75.000000     29.000000
max      1.905000e+08     91.000000     95.000000     44.000000 

Цена - возраст: 0.0
Гипотеза Н0 отвергается (цена не зависит от возраста) 

Цена - рейтинг: 0.0
Гипотеза Н0 отвергается (цена не зависит от рейтинга) 

Цена - возраст, потенциальный рейтинг: 0.0 , 0.0
Гипотеза Н0 не отвергается (цена одновременно зависит от потенциального рейтинга и возраста) 

Коэффициенты:
Свободный член: -39312206.65
Коэффициент для рейтинга, потенциального рейтинга и возраста: 698157.36 , 77438.21 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Задание 2